In [24]:
library(Seurat)
library(ggplot2)
library(dplyr)
library(patchwork)

# ========== 参数设置 ==========
prefix <- "08cot"
rds_path <- "/data/work/RNA_AraSeed/rds/08.Cot.rds"
output_dir <- "/data/work/RNA_AraSeed/rds/Ref_UMI_propotion"
col_classes <- "Cell_classes"
col_batch <- "Batch"

# ========== 颜色定义 ==========
default_class_colors <- c(
  "Embryo"     = "#43A047",
  "Endosperm"  = "#5BC0EB",
  "Seed Coat"  = "#F28C52",
  "Seed coat"  = "#F28C52",
  "Unknown"    = "#BDBDBD"
)

# ========== 读取数据 ==========
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)
setwd(output_dir)

sce <- readRDS(rds_path)
cat("Loaded:", ncol(sce), "cells,", nrow(sce), "genes\n")

# ========== 提取metadata ==========
meta <- sce@meta.data

# 检查列是否存在
if(!col_classes %in% colnames(meta)) stop(paste("列", col_classes, "不存在"))
if(!col_batch %in% colnames(meta)) stop(paste("列", col_batch, "不存在"))

# 设置因子
class_levels <- c("Embryo", "Endosperm", "Seed Coat", "Seed coat", "Unknown")
meta[[col_classes]] <- factor(meta[[col_classes]], levels = class_levels)
meta[[col_batch]] <- as.factor(meta[[col_batch]])

# ========== 构建绘图数据 ==========
plot_df <- data.frame(
  CellClass = meta[[col_classes]],
  Batch = meta[[col_batch]],
  UMI = meta$nCount_RNA,
  Genes = meta$nFeature_RNA
)
plot_df <- na.omit(plot_df)

# ========== Class级别Violin Plot (按Batch分组，按Class颜色) ==========
make_class_violin <- function(df, value_col, title_text, ylab_text = "Count (log10 scale)") {
  ggplot(df, aes(x = CellClass, y = .data[[value_col]], 
                 group = interaction(Batch, CellClass),
                 fill = CellClass)) +
    geom_violin(
      scale = "width",
      trim = TRUE,
      width = 0.9,
      color = "black",
      linewidth = 0.4,
      position = position_dodge(width = 0.9),
      draw_quantiles = c(0.25, 0.5, 0.75)
    ) +
    scale_fill_manual(values = default_class_colors, name = "Cell Class") +
    scale_y_log10() +
    labs(title = title_text, x = NULL, y = ylab_text) +
    theme_classic(base_size = 14) +
    theme(
      plot.title = element_text(hjust = 0.5, face = "bold", size = 16),
      axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1),
      axis.title.y = element_text(size = 13),
      legend.position = "right",
      legend.title = element_text(face = "bold")
    )
}

p_umi <- make_class_violin(plot_df, "UMI", paste0(prefix, " - UMIs per Class"))
p_genes <- make_class_violin(plot_df, "Genes", paste0(prefix, " - Genes per Class"))

fig_class <- p_umi / p_genes

pdf(paste0(prefix, "_class_violin.pdf"), width = 8, height = 10)
print(fig_class)
dev.off()

cat("\nSaved:", prefix, "_class_violin.pdf\n")

Loaded: 34871 cells, 33312 genes


pdf 
  2


Saved: 08cot _class_violin.pdf
